# 임베딩 모델 A/B 테스트

**목적:** 우리 데이터(지자체 복지 정책 574 + PDF 섹션 1,189)와 어르신 말투 질문에 가장 잘 맞는 **임베딩 모델**을 고르고, 같은 실험에서 **청킹 방식(A_fixed / B_structure)** 도 비교한다.

| | A_fixed (3,400) | B_structure (3,865) |
|---|---|---|
| qwen3-embedding:0.6b | ○ | ○ |
| bge-m3 | ○ | ○ |
| KURE-v1 | ○ | ○ |

- **모델 비교** = 같은 청킹 방식 안에서 / **청킹 비교** = 같은 모델 안에서 → 항상 하나만 다르게
- **똑같이 맞추는 것:** 같은 원본, 같은 질문·정답, k=5, COSINE, 같은 지역 필터, 청크 `text`만 임베딩

**평가 지표**

| 지표 | 뜻 |
|---|---|
| Hit@1 / Hit@3 / Hit@5 | k등 안에 정답이 하나라도 있는 비율 |
| Recall@5 | 정답이 여러 개인 질문에서 5등 안에 찾은 정답 수 ÷ 정답 수 |
| MRR@5 | 첫 정답 순위 점수 평균 (1등=1, 2등=0.5 … 5등 밖=0) |

**결정 규칙 (실험 전에 정함):** ① MRR@5 (A·B 평균) → ② 차이 0.02 미만이면 PDF근거 Hit@5 → ③ 그래도 비슷하면 무료·로컬·빠른 쪽. 청킹 방식은 선택한 모델에서 같은 규칙.

> 정답은 `parent_id`(정책 = policy_id, PDF = section_id)로 판정하므로 `chunk_id`가 다른 A와 B를 똑같이 채점할 수 있다.

## 1. 설정
- `CHUNK_SETS`: A·B 방식마다 **정책 파일 + PDF 파일** 2개
- `EXPECTED_COUNTS`: 인수인계 문서의 청크 수 → 읽은 개수와 비교
- `K = 5`: 팀 문서 지표(Hit@5 · Recall@5 · MRR@5)와 맞춤
- `SAMPLE_SIZE`: 처음엔 500 정도로 코드가 도는지만 확인, 본 테스트는 `None`

In [ ]:
CHUNK_SETS = {
    "A_fixed": [
        "../../data/rag/chunks/A_fixed/policy_chunks.jsonl",
        "../../data/rag/chunks/A_fixed/guideline_chunks.jsonl",
    ],
    "B_structure": [
        "../../data/rag/chunks/B_structure/policy_chunks.jsonl",
        "../../data/rag/chunks/B_structure/guideline_chunks.jsonl",
    ],
}
EXPECTED_COUNTS = {"A_fixed": 3400, "B_structure": 3865}
TEST_PATH = "./data/test_questions_v2.jsonl"

K = 5                                   # Hit@5, Recall@5, MRR@5
USE_REGION_FILTER = True                # 지역 질문에 지역 필터 (모든 모델 똑같이)
INCLUDE_TYPES = ("policy", "guideline")
SAMPLE_SIZE = None                      # 처음엔 500 정도로 빠르게, 본 테스트는 None

## 2. 청크 읽기
- 강사님 `JSONLoader` 사용
- 합의한 형식은 지역·출처가 `metadata` 안에 한 겹 더 들어 있어서 **펼쳐서** 읽는다
- 방식마다 정책 파일과 PDF 파일을 읽어서 합친다
- 개수가 인수인계 문서와 다르면 경고

In [ ]:
from collections import Counter
from langchain_community.document_loaders import JSONLoader

def metadata_func(record: dict, metadata: dict) -> dict:
    for key, value in record.items():
        if key == "text":
            continue
        if key == "metadata" and isinstance(value, dict):
            metadata.update(value)      # metadata 안의 정보를 펼치기
        else:
            metadata[key] = value
    return metadata

def load_documents(paths):
    docs = []
    for path in paths:                  # 정책 파일 + PDF 파일
        loader = JSONLoader(
            file_path=path,
            jq_schema=".",
            content_key="text",
            json_lines=True,
            metadata_func=metadata_func,
        )
        docs.extend(loader.load())
    docs = [d for d in docs if d.metadata.get("record_type", "policy") in INCLUDE_TYPES]
    if SAMPLE_SIZE:
        docs = docs[:SAMPLE_SIZE]
    return docs

chunk_data = {}
for chunk_name, paths in CHUNK_SETS.items():
    docs = load_documents(paths)
    chunk_data[chunk_name] = docs
    print(f"[{chunk_name}] 청크 수: {len(docs)} (기대 {EXPECTED_COUNTS[chunk_name]})",
          Counter(d.metadata.get("record_type") for d in docs))
    if not SAMPLE_SIZE and len(docs) != EXPECTED_COUNTS[chunk_name]:
        print("   ⚠️ 개수가 인수인계 문서와 다름! 파일 경로·버전 확인")

## 3. 테스트 질문 + 채점할 수 있는 질문만 고르기
- 질문 세트 v2 (40개): 지역정책 18 · 지역짝 6 · 시도전체 2 · PDF근거 10 · 함정 4
- 함정 질문은 정답이 없어서 제외 → **채점 36개**
- **정답(`parent_id`)이 데이터에 없는 질문도 제외** → 모델 탓이 아닌데 틀린 걸로 세어지는 걸 막기 위해

In [ ]:
import json

with open(TEST_PATH, encoding="utf-8") as f:
    all_questions = [json.loads(line) for line in f]

def select_questions(docs):
    parent_ids = {d.metadata.get("parent_id") for d in docs}   # 데이터에 있는 부모 번호

    questions, skipped = [], []
    for q in all_questions:
        if q["q_type"].startswith("함정"):
            continue                                            # 함정 질문은 정답이 없어서 제외
        ok = any(pid in parent_ids for pid in q["expected_ids"])
        (questions if ok else skipped).append(q)
    return questions, skipped

chunk_questions = {}
for chunk_name, docs in chunk_data.items():
    questions, skipped = select_questions(docs)
    chunk_questions[chunk_name] = questions
    print(f"[{chunk_name}] 채점할 질문: {len(questions)}개 / 정답이 데이터에 없어 제외: {len(skipped)}개")
    for q in skipped:
        print("   제외:", q["qid"], q["question"][:25], "| 정답:", q.get("expected_title"))

## 4. 비교할 모델
| 모델 | 고른 이유 |
|---|---|
| qwen3-embedding:0.6b | 수업 모델, 기준 |
| bge-m3 | 무료 다국어 대표 |
| KURE-v1 | bge-m3를 한국어로 추가 학습한 모델, 무료 |
| text-embedding-3-small | 유료 대표, 강사님 `ex02`에서 사용 |

딕셔너리에 한 줄만 추가하면 같은 테스트를 반복할 수 있다.

In [ ]:
from langchain_ollama import OllamaEmbeddings

models = {
    "qwen3-embedding:0.6b": OllamaEmbeddings(model="qwen3-embedding:0.6b"),
    "bge-m3": OllamaEmbeddings(model="bge-m3"),
}

# KURE-v1 (한국어 특화, 무료) / 설치: uv pip install langchain-huggingface sentence-transformers
# from langchain_huggingface import HuggingFaceEmbeddings
# models["KURE-v1"] = HuggingFaceEmbeddings(model_name="nlpai-lab/KURE-v1")

# OpenAI (유료) / .env에 OPENAI_API_KEY 필요
# from dotenv import load_dotenv; load_dotenv()
# from langchain_openai import OpenAIEmbeddings
# models["text-embedding-3-small"] = OpenAIEmbeddings(model="text-embedding-3-small")

## 5. 정답 판정 + 지역 필터
**지역 필터 (모든 모델에 똑같이)**
- 시도: 내 시도 **또는** 시도가 비어 있음 → PDF(전국 지침)는 `sido_name = null`이라 항상 포함
- 시군구: 내 시군구 **또는** 시군구가 비어 있음 → 시도 전체 대상 정책도 포함
- 같은 시도 안에 비슷한 정책(목욕비, 장수수당 등)이 많아서, 시군구까지 걸러야 **임베딩 실력만** 비교할 수 있다
- `IsEmpty` = 값이 null이거나 칸 자체가 없는 경우 (`IsNull`은 칸이 없으면 못 잡음)

**정답 판정**은 다음 셀에서 `parent_id`로 바로 비교한다.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, IsEmptyCondition, PayloadField

def match_or_empty(field, value):
    # 이 칸이 value이거나, 비어 있으면(null·칸 없음) 통과
    key = f"metadata.{field}"
    return Filter(should=[
        FieldCondition(key=key, match=MatchValue(value=value)),
        IsEmptyCondition(is_empty=PayloadField(key=key)),
    ])

def region_filter(q):
    if not USE_REGION_FILTER or not q.get("user_sido"):
        return None
    conditions = [match_or_empty("sido_name", q["user_sido"])]              # 내 시도 + 시도 없음(PDF 등)
    if q.get("user_sigungu"):
        conditions.append(match_or_empty("sigungu_name", q["user_sigungu"]))  # 내 시군구 + 시도 전체 정책
    return Filter(must=conditions)

## 6. 모델 하나 평가하기
- 강사님 방식: `QdrantVectorStore.from_documents` + `similarity_search_with_score`
- 테스트용이라 **메모리(`:memory:`)에만** 저장 → Docker Qdrant의 진짜 컬렉션은 건드리지 않음
- 적재 시간, 질문당 검색 시간도 기록
- 정답 판정: 결과 청크의 `parent_id`가 정답 목록에 있으면 정답. 같은 정책의 청크가 여러 개 나와도 **첫 순위만** 센다

In [ ]:
import time
from langchain_qdrant import QdrantVectorStore

def evaluate(model_name, embedding_model, chunk_name):
    docs = chunk_data[chunk_name]
    questions = chunk_questions[chunk_name]

    # 강사님 코드: from_documents (테스트라 메모리에만 저장)
    t0 = time.time()
    vectorstore = QdrantVectorStore.from_documents(
        documents=docs,
        embedding=embedding_model,
        location=":memory:",
        collection_name="ab_test",
    )
    build_sec = time.time() - t0

    hits = {1: 0, 3: 0, 5: 0}
    rr_sum = 0
    recall_sum = 0
    details = []

    t1 = time.time()
    for q in questions:
        results = vectorstore.similarity_search_with_score(
            q["question"], k=K, filter=region_filter(q)
        )

        expected = set(q["expected_ids"])
        rank = None          # 첫 정답 순위
        found = set()        # 찾은 정답 (같은 부모의 청크가 여러 개 나와도 한 번만)
        for i, (doc, score) in enumerate(results, start=1):
            pid = doc.metadata.get("parent_id")
            if pid in expected:
                found.add(pid)
                if rank is None:
                    rank = i

        for n in hits:
            if rank is not None and rank <= n:
                hits[n] += 1
        if rank is not None:
            rr_sum += 1 / rank
        recall_sum += len(found) / len(expected)

        first_doc, first_score = results[0] if results else (None, None)
        details.append({
            "chunk_set": chunk_name, "model": model_name, "qid": q["qid"],
            "q_type": q["q_type"], "question": q["question"],
            "rank": rank, "recall": round(len(found) / len(expected), 2),
            "top1_score": round(first_score, 3) if first_doc else None,
            "top1_title": first_doc.metadata.get("title") if first_doc else None,
        })

    total = len(questions)
    summary = {
        "chunk_set": chunk_name,
        "model": model_name,
        "dim": len(embedding_model.embed_query("테스트")),
        "Hit@1": round(hits[1] / total, 3),
        "Hit@3": round(hits[3] / total, 3),
        "Hit@5": round(hits[5] / total, 3),
        "Recall@5": round(recall_sum / total, 3),
        "MRR@5": round(rr_sum / total, 3),
        "적재(초)": round(build_sec, 1),
        "검색(초/질문)": round((time.time() - t1) / total, 2),
    }
    return summary, details

## 7. 전체 실행 + 결과표 (청킹 방식별, MRR@5 높은 순)
⏱ 모델 하나 × 방식 하나에 수 분~수십 분 걸릴 수 있다. 모델을 하나씩 돌려도 된다.

In [ ]:
import pandas as pd

summaries, all_details = [], []
for chunk_name in CHUNK_SETS:
    for model_name, embedding_model in models.items():
        print(f"평가 중: [{chunk_name}] {model_name}")
        summary, details = evaluate(model_name, embedding_model, chunk_name)
        summaries.append(summary)
        all_details.extend(details)

result_df = pd.DataFrame(summaries).sort_values(["chunk_set", "MRR@5"], ascending=[True, False])
result_df

## 8. 질문 유형별 정답률
지역정책 · 지역짝 · 시도전체 · PDF근거별로 5등 안에 정답이 나온 비율 → 모델의 약점 확인

In [ ]:
detail_df = pd.DataFrame(all_details)
detail_df["hit"] = detail_df["rank"].notna()      # 5등 안에 정답이 있었는지

detail_df.pivot_table(index=["chunk_set", "model"], columns="q_type", values="hit", aggfunc="mean").round(2)

## 9. 틀린 질문 보기 + 결과 저장
- 틀린 질문을 보면 모델의 약점을 알 수 있다
- CSV는 테스트 보고서에 그대로 사용 (`utf-8-sig` = 엑셀에서 한글 안 깨짐)

In [ ]:
import os

display(detail_df[~detail_df["hit"]][["chunk_set", "model", "qid", "question", "top1_title"]])

os.makedirs("./results", exist_ok=True)
result_df.to_csv("./results/ab_embedding_summary.csv", index=False, encoding="utf-8-sig")
detail_df.to_csv("./results/ab_embedding_detail.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

## 10. 결과 정리 (테스트 후 작성)
| 항목 | 내용 |
|---|---|
| 비교한 청킹 방식 | A_fixed / B_structure |
| 선택 모델 | |
| 벡터 길이 | |
| 선택한 청킹 방식 | |
| 선택 이유 | MRR@5 / Hit@5 / Recall@5 / 속도 / 비용 (결정 규칙 순서대로) |
| 약점 | (틀린 질문 유형) |

→ 다음: 결과 공유 → 청크 데이터 확정 → 선택한 모델 · 방식으로 본 임베딩 → Qdrant 적재. 질문 임베딩과 Neo4j도 같은 모델을 쓴다.